# Neon Web Traffic — Sample Logs Spectacle

Dashboard exported from example-mcp-dashbuilder


In [ ]:
from elasticsearch import Elasticsearch
import pandas as pd
import matplotlib.pyplot as plt

es = Elasticsearch(
    "http://localhost:9200",
    basic_auth=("elastic", "changeme"),
)


## Total requests

Chart type: **metric**


In [ ]:
result = es.esql.query(
    query="""
    FROM kibana_sample_data_logs | STATS lines = COUNT(*)
    """,
    format="json"
)

df = pd.DataFrame(result["values"], columns=[c["name"] for c in result["columns"]])
from IPython.display import HTML, display
value = df.iloc[0]['lines']
display(HTML(f'''
<div style="padding: 20px 24px; border: 1px solid #e0e0e0; border-radius: 8px; background: #fafafa; display: inline-block; min-width: 220px;">
  <div style="font-size: 13px; color: #666; text-transform: uppercase; letter-spacing: 0.5px; margin-bottom: 8px;">Total requests</div>
  <div style="font-size: 42px; font-weight: 600; color: #1a1a1a; line-height: 1;">{value}</div>
</div>
'''))


In [ ]:
# Total requests — trend sparkline
trend_result = es.esql.query(
    query="""
    FROM kibana_sample_data_logs | STATS lines = COUNT(*) BY ts = BUCKET(@timestamp, 1 day) | SORT ts
    """,
    format="json"
)

trend_df = pd.DataFrame(trend_result["values"], columns=[c["name"] for c in trend_result["columns"]])
trend_df.plot(title="Total requests — Trend")
plt.tight_layout()
plt.show()


## Bytes served

Chart type: **metric**


In [ ]:
result = es.esql.query(
    query="""
    FROM kibana_sample_data_logs | STATS mib = ROUND(SUM(bytes)/1048576, 1)
    """,
    format="json"
)

df = pd.DataFrame(result["values"], columns=[c["name"] for c in result["columns"]])
from IPython.display import HTML, display
value = df.iloc[0]['mib']
display(HTML(f'''
<div style="padding: 20px 24px; border: 1px solid #e0e0e0; border-radius: 8px; background: #fafafa; display: inline-block; min-width: 220px;">
  <div style="font-size: 13px; color: #666; text-transform: uppercase; letter-spacing: 0.5px; margin-bottom: 8px;">Bytes served</div>
  <div style="font-size: 42px; font-weight: 600; color: #1a1a1a; line-height: 1;">{value}</div>
</div>
'''))


## Geo destinations

Chart type: **metric**


In [ ]:
result = es.esql.query(
    query="""
    FROM kibana_sample_data_logs | STATS regions = COUNT_DISTINCT(geo.dest)
    """,
    format="json"
)

df = pd.DataFrame(result["values"], columns=[c["name"] for c in result["columns"]])
from IPython.display import HTML, display
value = df.iloc[0]['regions']
display(HTML(f'''
<div style="padding: 20px 24px; border: 1px solid #e0e0e0; border-radius: 8px; background: #fafafa; display: inline-block; min-width: 220px;">
  <div style="font-size: 13px; color: #666; text-transform: uppercase; letter-spacing: 0.5px; margin-bottom: 8px;">Geo destinations</div>
  <div style="font-size: 42px; font-weight: 600; color: #1a1a1a; line-height: 1;">{value}</div>
</div>
'''))


## HTTP error rate

Chart type: **metric**


In [ ]:
result = es.esql.query(
    query="""
    FROM kibana_sample_data_logs | STATS total = COUNT(*), errors = COUNT(*) WHERE TO_INTEGER(`response.keyword`) >= 400 | EVAL err_pct = ROUND(100.0 * errors / total, 2)
    """,
    format="json"
)

df = pd.DataFrame(result["values"], columns=[c["name"] for c in result["columns"]])
from IPython.display import HTML, display
value = df.iloc[0]['err_pct']
display(HTML(f'''
<div style="padding: 20px 24px; border: 1px solid #e0e0e0; border-radius: 8px; background: #fafafa; display: inline-block; min-width: 220px;">
  <div style="font-size: 13px; color: #666; text-transform: uppercase; letter-spacing: 0.5px; margin-bottom: 8px;">HTTP error rate</div>
  <div style="font-size: 42px; font-weight: 600; color: #1a1a1a; line-height: 1;">{value}</div>
</div>
'''))


## Traffic pulse (daily)

Chart type: **area**


In [ ]:
result = es.esql.query(
    query="""
    FROM kibana_sample_data_logs | STATS requests = COUNT(*) BY ts = BUCKET(@timestamp, 1 day) | SORT ts
    """,
    format="json"
)

df = pd.DataFrame(result["values"], columns=[c["name"] for c in result["columns"]])
df.plot.area(x="ts", y=["requests"], title="Traffic pulse (daily)")
plt.tight_layout()
plt.show()


## Top HTTP status codes

Chart type: **bar**


In [ ]:
result = es.esql.query(
    query="""
    FROM kibana_sample_data_logs | STATS hits = COUNT(*) BY code = response.keyword | SORT hits DESC | LIMIT 10
    """,
    format="json"
)

df = pd.DataFrame(result["values"], columns=[c["name"] for c in result["columns"]])
df.plot.bar(x="code", y=["hits"], title="Top HTTP status codes")
plt.tight_layout()
plt.show()


## Requests by OS (top 6)

Chart type: **pie**


In [ ]:
result = es.esql.query(
    query="""
    FROM kibana_sample_data_logs | STATS hits = COUNT(*) BY os = machine.os.keyword | SORT hits DESC | LIMIT 6
    """,
    format="json"
)

df = pd.DataFrame(result["values"], columns=[c["name"] for c in result["columns"]])
df.set_index("os")[["hits"]].plot.pie(subplots=True, title="Requests by OS (top 6)")
plt.tight_layout()
plt.show()


## When the internet glows (day × hour)

Chart type: **heatmap**


In [ ]:
result = es.esql.query(
    query="""
    FROM kibana_sample_data_logs | EVAL day = DATE_FORMAT("EEEE", @timestamp), hour = DATE_FORMAT("HH", @timestamp) | STATS hits = COUNT(*) BY day, hour | SORT day, hour
    """,
    format="json"
)

df = pd.DataFrame(result["values"], columns=[c["name"] for c in result["columns"]])
pivot = df.pivot_table(index="day", columns="hour", values="hits")
fig, ax = plt.subplots()
im = ax.imshow(pivot.values, cmap="YlOrRd", aspect="auto")
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=45, ha="right")
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        ax.text(j, i, f"{pivot.values[i, j]:.0f}", ha="center", va="center")
fig.colorbar(im, ax=ax)
ax.set_title("When the internet glows (day × hour)")
plt.tight_layout()
plt.show()
